In [4]:
import numpy as np
import pandas as pd

In [ ]:
# Generate e-commerce data set

n = 5000

customer_id = np.arange(1, n + 1)

age = np.random.normal(loc=38, scale=12, size=n).astype(int)
age = np.clip(age, 18, 70)

gender = np.random.choice(["Male", "Female"], size=n, p=[0.52, 0.48])

country = np.random.choice(
    ["USA", "India", "UK", "Canada", "Germany"],
    size=n,
    p=[0.45, 0.25, 0.12, 0.10, 0.08]
)

customer_segment = np.random.choice(
    ["New", "Regular", "Premium", "VIP"],
    size=n,
    p=[0.35, 0.40, 0.20, 0.05]
)

campaign = np.random.choice(
    ["Campaign_A", "Campaign_B"],
    size=n,
    p=[0.50, 0.50]
)

visit_frequency = np.random.poisson(lam=6, size=n)
visit_frequency = np.clip(visit_frequency, 1, 25)

discount = np.random.normal(loc=12, scale=6, size=n)
discount = np.clip(discount, 0, 40)

segment_effect = {
    "New": 20,
    "Regular": 45,
    "Premium": 90,
    "VIP": 180
}

campaign_effect = {
    "Campaign_A": 0,
    "Campaign_B": 12
}

base_purchase = []

for i in range(n):
    value = (
        25
        + age[i] * 0.8
        + visit_frequency[i] * 6
        + discount[i] * 1.5
        + segment_effect[customer_segment[i]]
        + campaign_effect[campaign[i]]
        + np.random.normal(0, 35)
    )
    base_purchase.append(value)

purchase_amount = np.array(base_purchase)


purchase_amount = np.exp(np.log(np.maximum(purchase_amount, 5)) + np.random.normal(0, 0.35, n))
purchase_amount = np.round(purchase_amount, 2)

quantity = np.random.poisson(lam=3, size=n)
quantity = np.clip(quantity, 1, 15)


return_probability = 0.05 + (discount / 100) + np.where(customer_segment == "New", 0.04, 0)
returned = np.random.binomial(1, np.clip(return_probability, 0, 0.45))


churn_probability = (
    0.35
    - visit_frequency * 0.015
    - np.log1p(purchase_amount) * 0.025
    + np.where(customer_segment == "New", 0.10, 0)
)
churn_probability = np.clip(churn_probability, 0.02, 0.65)
churned = np.random.binomial(1, churn_probability)

df = pd.DataFrame({
    "CustomerID": customer_id,
    "Age": age,
    "Gender": gender,
    "Country": country,
    "CustomerSegment": customer_segment,
    "Campaign": campaign,
    "VisitFrequency": visit_frequency,
    "Discount": np.round(discount, 2),
    "Quantity": quantity,
    "PurchaseAmount": purchase_amount,
    "Returned": returned,
    "Churned": churned
})

print("Dataset shape:", df.shape)
print(df.head())


Dataset shape: (5000, 12)
   CustomerID  Age  Gender Country CustomerSegment    Campaign  \
0           1   39    Male     USA             New  Campaign_A   
1           2   24  Female     USA             New  Campaign_A   
2           3   50    Male     USA         Regular  Campaign_A   
3           4   27  Female  Canada             New  Campaign_A   
4           5   42    Male   India         Regular  Campaign_A   

   VisitFrequency  Discount  Quantity  PurchaseAmount  Returned  Churned  
0               5     12.00         1          121.88         0        0  
1               6      0.00         4           47.60         0        0  
2               4     15.74         3          110.48         0        0  
3               5      5.72         2           45.75         0        0  
4               1     15.45         4          136.71         0        0  


In [6]:
# Data Overview and Cleaning

print("\nData Info:")
print(df.info())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nBasic description:")
print(df.describe())



Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CustomerID       5000 non-null   int64  
 1   Age              5000 non-null   int64  
 2   Gender           5000 non-null   object 
 3   Country          5000 non-null   object 
 4   CustomerSegment  5000 non-null   object 
 5   Campaign         5000 non-null   object 
 6   VisitFrequency   5000 non-null   int32  
 7   Discount         5000 non-null   float64
 8   Quantity         5000 non-null   int32  
 9   PurchaseAmount   5000 non-null   float64
 10  Returned         5000 non-null   int32  
 11  Churned          5000 non-null   int32  
dtypes: float64(2), int32(4), int64(2), object(4)
memory usage: 390.8+ KB
None

Missing values:
CustomerID         0
Age                0
Gender             0
Country            0
CustomerSegment    0
Campaign           0
VisitFrequency 

In [7]:
# Descriptive statistics


numeric_cols = ["Age", "VisitFrequency", "Discount", "Quantity", "PurchaseAmount"]

desc_stats = df[numeric_cols].agg([
    "mean",
    "median",
    "std",
    "var",
    "min",
    "max",
    "skew",
    "kurt"
]).T

desc_stats["range"] = df[numeric_cols].max() - df[numeric_cols].min()
desc_stats["IQR"] = df[numeric_cols].quantile(0.75) - df[numeric_cols].quantile(0.25)

print("\nDescriptive Statistics:")
print(desc_stats)



Descriptive Statistics:
                      mean  median        std          var   min     max  \
Age              37.743600   38.00  11.349768   128.817222  18.0   70.00   
VisitFrequency    5.963800    6.00   2.377061     5.650420   1.0   17.00   
Discount         12.066206   11.87   5.946680    35.362999   0.0   33.20   
Quantity          3.074600    3.00   1.657946     2.748785   1.0   12.00   
PurchaseAmount  175.853940  157.62  88.167124  7773.441782   3.1  770.46   

                    skew      kurt   range       IQR  
Age             0.188973 -0.429362   52.00   16.0000  
VisitFrequency  0.440916  0.240949   16.00    3.0000  
Discount        0.155055 -0.263117   33.20    8.2300  
Quantity        0.744375  0.381831   11.00    2.0000  
PurchaseAmount  1.322292  2.809317  767.36  104.4225  
